import pandas as pd

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet("C:/Users/okial/Desktop/DST-AIRLINES/data/flights/silver/atl_jfk_flights_clean.parquet")

#df = df[df['CANCELLED'] == 1]

df = df[df['OP_CARRIER'] == 'B6']
df 

In [ ]:
df['FL_DATE'] = pd.to_datetime(
                                            df['FL_DATE'],
                                            format='%Y-%m-%d',
                                            errors='coerce'
                                        )

print(df.head())

In [ ]:
df = pd.read_parquet("C:/Users/okial/Desktop/DST-AIRLINES/data/airports/silver/dim_airport.parquet")
df[df['IATA_CODE'] == 'ATL']

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airlines.dat"

COLS = [
    "airline_id", "name", "alias", "iata",
    "icao", "callsign", "country", "active",
]

def load_openflights_airlines():
    df = pd.read_csv(
        URL,
        header=None,
        names=COLS,
        na_values=["\\N", "", "-"],
        keep_default_na=True,
        encoding="utf-8",
    )

    # On ne garde que ce qui a un code IATA exploitable (2 caractères)
    df = df[df["iata"].notna()]
    df = df[df["iata"].str.len() == 2]

    # DIM_AIRLINE : mapping code IATA -> nom
    dim_airline = (
        df[["iata", "name"]]
        .dropna(subset=["name"])
        .drop_duplicates(subset=["iata"], keep="first")
        .rename(columns={"iata": "iata_code", "name": "airline_name"})
        .sort_values("iata_code")
        .reset_index(drop=True)
    )

    return dim_airline


if __name__ == "__main__":
    dim = load_openflights_airlines()
    print(dim.shape)
    print(dim.head(20))
    dim.to_parquet("dim_airline.parquet", index=False)

In [ ]:
import pandas as pd
df = pd.read_csv("C:\\Users\\okial\\Desktop\\DST-AIRLINES\\transform\\dim_airline.csv")
df[df['iata_code'].str.len() == max(df['iata_code'].str.len())]

In [ ]:
import pandas as pd

LOOKUP_PATH = "../data/airlines/L_UNIQUE_CARRIERS.csv"

def build_dim_airline(lookup_path=LOOKUP_PATH):
    df = pd.read_csv(lookup_path, encoding="utf-8")

    # BTS livre les colonnes "Code" et "Description"
    dim_airline = (
        df.rename(columns={"Code": "iata_code", "Description": "airline_name"})
        [["iata_code", "airline_name"]]
        .dropna()
        .drop_duplicates(subset=["iata_code"])
        .sort_values("iata_code")
        .reset_index(drop=True)
    )

    return dim_airline


if __name__ == "__main__":
    dim = build_dim_airline()
    print(dim.shape)
    print(dim.head(20))
    dim.to_csv("dim_airline.csv", index=False)

In [ ]:
df = pd.read_parquet("..\\data\\aircafts\\silver\\dim_aircraft.parquet")
#df = df[df['TAIL_NUM'] == 'N235WN']
df

In [ ]:
import pandas as pd, glob
tails = set()
for f in glob.glob("C:/Users/okial/Desktop/DST-AIRLINES/data/flights/silver/atl_jfk_flights_clean.parquet"):
    s = pd.read_parquet(f, columns=["TAIL_NUM"])["TAIL_NUM"].dropna().astype(str).str.strip().str.upper()
    tails.update(s[s != ""])
dim = pd.read_parquet("..\\data\\aircafts\\silver\\dim_aircraft.parquet")
print(sorted(tails - set(dim["TAIL_NUM"])))

In [ ]:
import pandas as pd
df = pd.read_parquet( "..\data\weather\dim_weather_2.parquet")

print(df[["snow", "wpgt"]].notna().sum())

df # taux de complétude par colonne

#dups = df.duplicated(subset=["AIRPORT_IATA_CODE", "WEATHER_TIME_UTC"], keep=False)
#print(df[dups].sort_values(["AIRPORT_IATA_CODE", "WEATHER_TIME_UTC"]).head(20))

In [ ]:
import pandas as pd

df =  pd.read_parquet("..\\data\\lookups\\station_map.parquet")
df

In [ ]:
import os
from dotenv import load_dotenv

print("Répertoire courant :", os.getcwd())

load_dotenv()

print("Clé trouvée :", os.getenv("OPENWEATHER_API_KEY") )

In [ ]:
df = pd.read_parquet("C:\\Users\\okial\\Desktop\\DST-AIRLINES\\extract\\data\\weather\\weather_realtime.parquet")
df

In [ ]:
import requests 
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

lat = 33.6367
lon = -84.4281
params = {
        "lat": lat, "lon": lon,
        "units": "metric",                       # temp en °C, vent en m/s
        "appid": API_KEY,
    }
resp = requests.get(BASE_URL, params=params, timeout=15)
resp.raise_for_status()
data = resp.json()
data

In [ ]:
import pandas as pd
df = pd.read_parquet("C:\\Users\\okial\\Desktop\\DST-AIRLINES\\data\\flights\\silver\\bts_2026_05.parquet")
df
print(df['ORIGIN'].value_counts())       # répartition ATL / JFK
print(df['FL_DATE'].nunique())           # combien de jours couverts ?
print(df.groupby('FL_DATE').size())      # vols par jour
print(df['CANCELLED'].value_counts())    # dont annulés

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.concat([pd.read_parquet(p) for p in sorted(Path("../data/flights/silver").glob("bts_*.parquet"))])
df.to_parquet("../data/flights/silver/bts_all.parquet", index=False)